# Blinded pharmacogenomics study

Genotype description -> diplotype -> CPIC phenotype -> CPIC Level A recommendation, for 110
blinded cases, using only `cases_blinded.json` and `cpic_guideline_corpus.json`.

No network access. The pipeline lives in `pgx_pipeline.py`; this notebook runs it and
summarises the result. See `METHODS_NOTE.md` for the interpretation decisions.

In [ ]:
import json, os, sys
import pandas as pd

sys.path.append(os.getcwd())      # the pipeline module sits next to this notebook
import pgx_pipeline as P

DIR = os.path.dirname(os.path.abspath("pgx_pipeline.py"))
cases = json.load(open(os.path.join(DIR, "cases_blinded.json")))
corpus = json.load(open(os.path.join(DIR, "cpic_guideline_corpus.json")))
print(len(cases), "cases |", len(corpus["genes"]), "genes |", corpus["schema_version"],
      corpus["generated_at_utc"])

## 1. Corpus coverage check

Every gene-drug pair in the case set must appear in the corpus before anything is answered.

In [ ]:
cs = pd.DataFrame(cases)
cov = {g: {d.lower() for d in v["drugs_covered"]} for g, v in corpus["genes"].items()}
chk = (cs.groupby(["gene", "drug"]).size().rename("n_cases").reset_index())
chk["in_corpus"] = [r.gene in cov and r.drug.lower() in cov[r.gene] for r in chk.itertuples()]
chk["corpus_sources"] = [";".join(s["source_org"] for s in corpus["genes"][r.gene]["sources"]
                                  if any(r.drug.lower() in x.lower()
                                         for x in s.get("drug_scope", [])))
                         for r in chk.itertuples()]
assert chk.in_corpus.all()
chk

## 2. Run the pipeline

`P.run` writes `claude_science_answers.csv`, `case_audit_trail.csv`,
`allele_definitions.csv`, `diplotype_to_phenotype.csv`,
`phenotype_to_recommendation.csv` and `run_provenance.json`.

In [ ]:
rows, audit, prov = P.run(DIR, DIR)
prov

## 3. Validate the answers file

In [ ]:
ans = pd.read_csv(os.path.join(DIR, "claude_science_answers.csv"))
assert list(ans.columns) == ["case_id", "diplotype", "phenotype", "recommendation", "abstained"]
assert len(ans) == len(cases)
assert list(ans.case_id) == [c["id"] for c in cases]
assert ans.case_id.nunique() == len(cases)
assert ans.notna().all().all()
assert set(ans.abstained.unique()) <= {True, False}
print("OK:", ans.shape, "|", int(ans.abstained.sum()), "abstentions")
ans.head()

## 4. Per-case audit trail

One row per case with every intermediate value, so any answer can be traced from the raw
genotype string to the recommendation.

In [ ]:
ad = pd.DataFrame(audit)
ad[["case_id", "gene", "drug", "genotype", "diplotype", "activity_score", "phenotype",
    "harm_class", "abstained"]]

## 5. Summaries

In [ ]:
print("phenotype classes assigned")
print(ad.pheno_key.value_counts().to_string())
print()
print("harm class")
print(ad.harm_class.value_counts().to_string())
print()
print("lethal-class cases by gene")
print(ad[ad.harm_class == "lethal"].groupby(["gene", "drug"]).size().to_string())

In [ ]:
# cases carrying an explicit interpretation assumption
ad.loc[ad.note.str.len() > 0, ["case_id", "gene", "drug", "diplotype", "note"]]

## 6. Lookup tables

In [ ]:
pd.read_csv(os.path.join(DIR, "allele_definitions.csv"))

In [ ]:
pd.read_csv(os.path.join(DIR, "diplotype_to_phenotype.csv"))

In [ ]:
p2r = pd.read_csv(os.path.join(DIR, "phenotype_to_recommendation.csv"))
print(len(p2r), "recommendation rows")
p2r

## 7. Regenerate the methods note

In [ ]:
import build_methods_note as B  # noqa: E402
_ = B.build(DIR, DIR)
print("METHODS_NOTE.md written")